In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor,CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

In [2]:

print("Loading dataset for training...")
df = pd.read_csv("smart_traffic_management_bengaluru_14days_30min_CORRECTED.csv")


Loading dataset for training...


In [10]:


df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['day_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7.0)
df['day_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7.0)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
categorical_vars = ['weather_condition']

In [12]:
vol_features = [
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 
    'latitude', 'longitude', 'weather_condition', 
    'temperature', 'humidity', 'accident_reported'
]
cat_vol = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=False)
cat_vol.fit(train_df[vol_features], train_df['number_of_vehicles'], cat_features=categorical_vars)
vol_preds = cat_vol.predict(test_df[vol_features])
print(f"1. Volume Model (Regressor) -> R2 Score: {r2_score(test_df['number_of_vehicles'], vol_preds):.4f} | RMSE: {np.sqrt(mean_squared_error(test_df['number_of_vehicles'], vol_preds)):.2f}")
cat_vol.fit(df[vol_features], df['number_of_vehicles'], cat_features=categorical_vars) # Retrain on 100% for saving
cat_vol.save_model("traffic_model.cbm")

1. Volume Model (Regressor) -> R2 Score: 0.9571 | RMSE: 19.21


In [11]:
print("Training Congestion Level Classifier...")
congestion_features = [
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 
    'latitude', 'longitude', 'weather_condition', 
    'number_of_vehicles', 'accident_reported'
]

cat_congestion = CatBoostClassifier(iterations=300, depth=6, random_seed=42, verbose=False)
cat_congestion.fit(train_df[congestion_features], train_df['congestion_level'], cat_features=categorical_vars)

cong_preds = cat_congestion.predict(test_df[congestion_features])
print(f"4. Congestion Level Model (Classifier) -> Accuracy: {accuracy_score(test_df['congestion_level'], cong_preds)*100:.2f}%")

cat_congestion.fit(df[congestion_features], df['congestion_level'], cat_features=categorical_vars)
cat_congestion.save_model("congestion_model.cbm")

Training Congestion Level Classifier...
4. Congestion Level Model (Classifier) -> Accuracy: 85.50%
